In [1]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


Device: cuda
  GPU : NVIDIA GeForce RTX 4090 Laptop GPU
  VRAM: 17.2 GB


In [2]:
import sys, subprocess

def check(pkg, install_name=None):
    try:
        __import__(pkg)
        import importlib
        ver = getattr(importlib.import_module(pkg), '__version__', 'OK')
        print(f'  ok  {pkg}: {ver}')
    except ImportError:
        name = install_name or pkg
        print(f'  installing {name} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', name])
        print(f'  ok  {pkg}')

for p, i in [('torch', None), ('torchvision', None), ('numpy', None),
             ('matplotlib', None), ('seaborn', None), ('sklearn', 'scikit-learn'),
             ('joblib', None), ('tqdm', None), ('kneed', None)]:
    check(p, i)
print('All packages ready.')


  ok  torch: 2.5.1+cu121
  ok  torchvision: 0.20.1+cu121
  ok  numpy: 1.26.4
  ok  matplotlib: 3.8.0
  ok  seaborn: 0.13.0
  ok  sklearn: 1.3.2
  ok  joblib: 1.3.2
  ok  tqdm: 4.66.1
  ok  kneed: 0.8.6
All packages ready.


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import os, gc, time, pickle, json as _json
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

torch.backends.cudnn.benchmark = True
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


In [4]:
from datetime import datetime
from pathlib import Path

dt_str = datetime.now().strftime('%y%m%d_%H%M')
print(dt_str)

# ── Settings ─────────────────────────────────────────────────────────────────
# source_name  = 'culture_LCC'
# model_name   = 'culture'
# model_name2  = 'culture'

source_name  = 'culture_LCC'
model_name   = 'culture-noAug'
model_name2  = 'SimCLR_culture'

#source_name  = 'tissue_FTC'
#model_name   = 'tissue'
#model_name2  = 'tissue'

IMAGE_SIZE   = 150        
BATCH_SIZE   = 64
EPOCHS       = 3
LR           = 1e-4    
WEIGHT_DECAY = 1e-4
FEAT_DIM     = 128        
HEAD_HIDDEN  = 512        
HEAD_BOTTLE  = 128        
OUT_DIM      = 256        

# Multi-crop settings
N_GLOBAL     = 2          
N_LOCAL      = 2          
N_CROPS      = N_GLOBAL + N_LOCAL

# DINO temperature settings
STUDENT_TEMP          = 0.1
TEACHER_TEMP          = 0.04   
TEACHER_TEMP_WARMUP   = 0.04   
TEACHER_TEMP_WARMUP_E = 30     

# EMA momentum (cosine schedule: start → end)
MOMENTUM_START = 0.996  
MOMENTUM_END   = 1.0

# Center momentum
CENTER_MOMENTUM = 0.9997  

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE        = f'./Numpy/{model_name}/{model_name2}'

RESULT_DIR  = f'./result_dir/{dt_str}_dino'

import glob as _g, os as _o

Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)

# ── Resume settings ──────────────────────────────────────────────────────────
RESUME_CHECKPOINT = _o.path.join(RESULT_DIR, 'checkpoint_latest.pth')  # resume patch
CHECKPOINT_EVERY  = 1     
GRAD_CLIP         = 1.0    

print(f'RESULT_DIR : {RESULT_DIR}')
print(f'OUT_DIM    : {OUT_DIM}  (HEAD_BOTTLE={HEAD_BOTTLE})')
print(f'N_CROPS    : {N_CROPS} (global={N_GLOBAL}, local={N_LOCAL})')
print(f'TEACHER_TEMP: {TEACHER_TEMP}')


261106_2034
RESULT_DIR : ./result_dir/261106_2034_dino
OUT_DIM    : 256  (HEAD_BOTTLE=128)
N_CROPS    : 4 (global=2, local=2)
TEACHER_TEMP: 0.04


In [5]:
class MultiCropAugmentation:

    def __init__(self, size=150, n_global=2, n_local=2):
        self.n_global = n_global
        self.n_local  = n_local
        norm = T.Normalize(mean=[0.5], std=[0.5])
        ks   = max(3, size // 20 * 2 + 1)

        self.global_transform = T.Compose([
            T.RandomResizedCrop(size, scale=(0.4, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(180),
            T.GaussianBlur(ks, sigma=(0.1, 2.0)),
            norm,
        ])
        self.local_transform = T.Compose([
            T.RandomResizedCrop(size, scale=(0.05, 0.4)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            norm,
        ])

    def __call__(self, img):
        crops = []
        for _ in range(self.n_global):
            crops.append(self.global_transform(img).repeat(3, 1, 1))
        for _ in range(self.n_local):
            crops.append(self.local_transform(img).repeat(3, 1, 1))
        return crops  


class TensorImageDataset(torch.utils.data.Dataset):
 
    def __init__(self, X_cache, transform):
        self.X         = X_cache  
        self.transform = transform

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        return self.transform(self.X[idx])  


In [6]:
print('Loading data ...')
X_train = np.load(f'{BASE}_X_train.npy')
y_train = np.load(f'{BASE}_y_train.npy')
X_val   = np.load(f'{BASE}_X_val.npy')
y_val   = np.load(f'{BASE}_y_val.npy')

X_all = np.concatenate([X_train, X_val], axis=0)
y_all = np.concatenate([y_train, y_val], axis=0)
print(f'X_all: {X_all.shape},  y_all: {y_all.shape}')

X_arr = X_all.squeeze(-1) if X_all.ndim == 4 else X_all  
X_arr = X_arr.astype(np.float32)
if X_arr.max() > 1.0:
    X_arr /= 255.0
X_cache = torch.from_numpy(X_arr).unsqueeze(1)          
print(f'Image cache: {X_cache.shape},  {X_cache.nbytes/1e6:.1f} MB on CPU RAM')

# ── DataLoader  ───────────────────────────────────────────────────
import platform
n_workers = 0 if platform.system() == 'Windows' else 4

multi_crop = MultiCropAugmentation(IMAGE_SIZE, N_GLOBAL, N_LOCAL)
dataset    = TensorImageDataset(X_cache, multi_crop)
loader     = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=n_workers,
                        pin_memory=(device.type == 'cuda' and n_workers > 0),
                        persistent_workers=(n_workers > 0),
                        drop_last=True)
print(f'DataLoader: {len(loader)} batches/epoch,  num_workers={n_workers}  ({platform.system()})')


Loading data ...
X_all: (3560, 150, 150, 1),  y_all: (3560,)
Image cache: torch.Size([3560, 1, 150, 150]),  320.4 MB on CPU RAM
DataLoader: 55 batches/epoch,  num_workers=0  (Windows)


In [7]:
# ─── CNN Backbone ─────────────────────────────────────────────────────────────
class CNNBackbone(nn.Module):
    
    def __init__(self):
        super().__init__()
        def block(ci, co, pool=True):
            layers = [nn.Conv2d(ci, co, 3, padding=1), nn.ReLU(True),
                      nn.BatchNorm2d(co),
                      nn.Conv2d(co, co, 3, padding=1), nn.ReLU(True),
                      nn.BatchNorm2d(co)]
            if pool: layers.append(nn.MaxPool2d(2))
            return nn.Sequential(*layers)

        self.encoder = nn.Sequential(
            block(3, 64), block(64, 128), block(128, 256),
            block(256, 512), block(512, 512),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(True),
            nn.Linear(256, FEAT_DIM),
        )

    def forward(self, x):
        return self.fc(self.pool(self.encoder(x)))

# ─── DINO Projection Head ─────────────────────────────────────────────────────
class DINOHead(nn.Module):
  
    def __init__(self, in_dim=FEAT_DIM, hidden_dim=HEAD_HIDDEN,
                 bottleneck_dim=HEAD_BOTTLE, out_dim=OUT_DIM):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, bottleneck_dim),
        )
        self.last_layer = nn.Linear(bottleneck_dim, out_dim, bias=False)
        nn.init.normal_(self.last_layer.weight, std=0.1)  

    def forward(self, x):
        x = self.mlp(x)
        x = F.normalize(x, dim=-1, p=2)       
        return self.last_layer(x)             

# ─── Student / Teacher instantiation ─────────────────────────────────────────
student_backbone = CNNBackbone().to(device)
student_head     = DINOHead().to(device)

teacher_backbone = CNNBackbone().to(device)
teacher_backbone.load_state_dict(student_backbone.state_dict())
teacher_head = DINOHead().to(device)
teacher_head.load_state_dict(student_head.state_dict())

for p in teacher_backbone.parameters(): p.requires_grad_(False)
for p in teacher_head.parameters():     p.requires_grad_(False)

s_params = sum(p.numel() for p in student_backbone.parameters() if p.requires_grad)
h_params = sum(p.numel() for p in student_head.parameters()     if p.requires_grad)
print(f'Student backbone params : {s_params:,}')
print(f'Student head    params  : {h_params:,}')
print(f'OUT_DIM = {OUT_DIM},  HEAD_BOTTLE = {HEAD_BOTTLE}')


Student backbone params : 9,575,104
Student head    params  : 427,136
OUT_DIM = 256,  HEAD_BOTTLE = 128


In [8]:
# ─── DINO Loss ────────────────────────────────────────────────────────────────
class DINOLoss(nn.Module):
    def __init__(self, out_dim, n_crops, n_global,
                 warmup_teacher_temp, teacher_temp, warmup_epochs, epochs,
                 student_temp=0.1, center_momentum=0.9):
        super().__init__()
        self.student_temp    = student_temp
        self.center_momentum = center_momentum
        self.n_crops         = n_crops
        self.n_global        = n_global
        self.register_buffer('center', torch.zeros(1, out_dim))

        self.teacher_temp_schedule = np.concatenate([
            np.linspace(warmup_teacher_temp, teacher_temp, warmup_epochs),
            np.ones(max(0, epochs - warmup_epochs)) * teacher_temp,
        ])

    def forward(self, student_out_list, teacher_out_list, epoch):
        t_temp = self.teacher_temp_schedule[min(epoch, len(self.teacher_temp_schedule)-1)]
        teacher_probs = [
            F.softmax((t - self.center) / t_temp, dim=-1).detach()
            for t in teacher_out_list
        ]
        total_loss, n_terms = 0.0, 0
        for iq, q in enumerate(teacher_probs):
            for v in range(self.n_crops):
                if v == iq: continue
                s = F.log_softmax(student_out_list[v] / self.student_temp, dim=-1)
                total_loss += (-q * s).sum(dim=-1).mean()
                n_terms    += 1
        loss = total_loss / n_terms
        self._update_center(teacher_out_list)
        return loss

    @torch.no_grad()
    def _update_center(self, teacher_out_list):
        all_t = torch.cat(teacher_out_list, dim=0).float()
        batch_center = all_t.mean(dim=0, keepdim=True)
        self.center = (self.center.float() * self.center_momentum
                       + batch_center * (1 - self.center_momentum))


# ─── Optimizer ────────────────────────────────────────────────────────────────
dino_loss = DINOLoss(
    out_dim=OUT_DIM, n_crops=N_CROPS, n_global=N_GLOBAL,
    warmup_teacher_temp=TEACHER_TEMP_WARMUP,
    teacher_temp=TEACHER_TEMP,
    warmup_epochs=TEACHER_TEMP_WARMUP_E,
    epochs=EPOCHS,
    student_temp=STUDENT_TEMP,
    center_momentum=CENTER_MOMENTUM,
).to(device)

_trainable = (list(student_backbone.parameters())
              + list(student_head.parameters()))
optimizer = torch.optim.AdamW(_trainable, lr=LR, weight_decay=WEIGHT_DECAY, eps=1e-6) 
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=100, eta_min=1e-6) 

momentum_schedule = np.linspace(MOMENTUM_START, MOMENTUM_END, EPOCHS * len(loader))

use_amp = (device.type == 'cuda')
scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

print('Loss / Optimizer / AMP ready.')
print(f'  Steps per epoch : {len(loader)}')
print(f'  Total steps     : {EPOCHS * len(loader):,}')
print(f'  AMP enabled     : {use_amp}')


Loss / Optimizer / AMP ready.
  Steps per epoch : 55
  Total steps     : 165
  AMP enabled     : True


C:\TEMP\ipykernel_59220\2136924447.py:63: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)


In [9]:
# ─── Phase 1: DINO training ───────────────────────────────
import matplotlib
matplotlib.use('Agg')

def save_loss_curve(loss_history, save_path, total_epochs):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(1, len(loss_history)+1), loss_history,
            color='royalblue', linewidth=1.8, label='Train loss')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('DINO Loss', fontsize=12)
    ax.set_title('DINO Pre-training Loss Curve (v2)', fontsize=14)
    ax.set_xlim(1, max(total_epochs, len(loss_history)))
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

start_epoch  = 1
global_step  = 0
loss_history = []
best_loss    = float('inf')

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    
    ckpt = torch.load(RESUME_CHECKPOINT, map_location=device)
    student_backbone.load_state_dict(ckpt['student_backbone_state'])
    student_head.load_state_dict(ckpt['student_head_state'])
    teacher_backbone.load_state_dict(ckpt['teacher_backbone_state'])
    teacher_head.load_state_dict(ckpt['teacher_head_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    if use_amp and 'scaler_state' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state'])
    loss_history = ckpt['loss_history']
    best_loss    = ckpt['best_loss']
    global_step  = ckpt['global_step']
    start_epoch  = ckpt['epoch'] + 1
    
else:
    print('start training')

print(f"{'Epoch':>6} {'Loss':>10} {'Δ Loss':>10} {'LR':>10}  {'Best':>10}")
print("-" * 56)

for epoch in range(start_epoch, EPOCHS + 1):
    student_backbone.train()
    student_head.train()
    teacher_backbone.eval()
    teacher_head.eval()

    epoch_loss   = 0.0
    running_loss = 0.0
    LOG_EVERY    = 10

    pbar = tqdm(loader, desc=f'Ep {epoch:3d}/{EPOCHS}',
                leave=False, dynamic_ncols=True)

    for step, batch_crops in enumerate(pbar):
        crops = [c.to(device) for c in batch_crops]

        with torch.cuda.amp.autocast(enabled=use_amp):
            student_feats = [student_backbone(c) for c in crops]
            student_outs  = [student_head(f)     for f in student_feats]
            with torch.no_grad():
                teacher_feats = [teacher_backbone(crops[i]) for i in range(N_GLOBAL)]
                teacher_outs  = [teacher_head(f)            for f in teacher_feats]

        loss = dino_loss(student_outs, teacher_outs, epoch - 1)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(
            list(student_backbone.parameters()) + list(student_head.parameters()),
            max_norm=GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        # EMA Teacher update
        m = momentum_schedule[min(global_step, len(momentum_schedule) - 1)]
        with torch.no_grad():
            for sp, tp in zip(student_backbone.parameters(),
                               teacher_backbone.parameters()):
                tp.data = m * tp.data + (1 - m) * sp.data
            for sp, tp in zip(student_head.parameters(),
                               teacher_head.parameters()):
                tp.data = m * tp.data + (1 - m) * sp.data

        loss_val     = loss.item()
        epoch_loss  += loss_val
        running_loss += loss_val
        global_step += 1

        if (step + 1) % LOG_EVERY == 0 or (step + 1) == len(loader):
            avg_running = running_loss / min(LOG_EVERY, step + 1)
            pbar.set_postfix({
                'loss' : f'{loss_val:.4f}',
                'avg10': f'{avg_running:.4f}',
                'mom'  : f'{m:.4f}',
            })
            running_loss = 0.0

    pbar.close()

    avg_loss = epoch_loss / len(loader)
    loss_history.append(avg_loss)
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    delta     = avg_loss - loss_history[-2] if len(loss_history) >= 2 else 0.0
    arrow     = '↓' if delta < -1e-6 else ('↑' if delta > 1e-6 else '─')
    is_best   = avg_loss < best_loss
    if is_best:
        best_loss = avg_loss
    best_mark = ' ★' if is_best else ''
    print(f'{epoch:6d} {avg_loss:10.4f} {delta:+10.4f} {lr_now:10.2e}  {best_loss:10.4f}{best_mark} {arrow}')

    if epoch % 10 == 0 or epoch == EPOCHS:
        save_loss_curve(loss_history, f'{RESULT_DIR}/dino_loss_curve.tif', EPOCHS)

    if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS:
        ckpt_data = {
            'epoch'                  : epoch,
            'student_backbone_state' : student_backbone.state_dict(),
            'student_head_state'     : student_head.state_dict(),
            'teacher_backbone_state' : teacher_backbone.state_dict(),
            'teacher_head_state'     : teacher_head.state_dict(),
            'optimizer_state'        : optimizer.state_dict(),
            'scheduler_state'        : scheduler.state_dict(),
            'scaler_state'           : scaler.state_dict() if use_amp else None,
            'loss_history'           : loss_history,
            'best_loss'              : best_loss,
            'global_step'            : global_step,
        }
        torch.save(ckpt_data, f'{RESULT_DIR}/checkpoint_latest.pth')
        if is_best:
            torch.save(ckpt_data, f'{RESULT_DIR}/checkpoint_best.pth')

print()
print('Pre-training complete.')
print(f'Best loss : {best_loss:.4f}  (epoch {int(np.argmin(loss_history))+1})')


start training
 Epoch       Loss     Δ Loss         LR        Best
--------------------------------------------------------


Ep   1/3:   0%|                                                                                 | 0/55 [00:00<…

C:\TEMP\ipykernel_59220\2281671041.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


     1     3.1823    +0.0000   1.00e-04      3.1823 ★ ─


Ep   2/3:   0%|                                                                                 | 0/55 [00:00<…

     2     2.6773    -0.5050   9.99e-05      2.6773 ★ ↓


Ep   3/3:   0%|                                                                                 | 0/55 [00:00<…

     3     2.4738    -0.2035   9.98e-05      2.4738 ★ ↓

Pre-training complete.
Best loss : 2.4738  (epoch 3)


In [10]:
# ─── Phase 2: Feature extraction (128-dim backbone output) ────────────────────

def extract_features(backbone, X, batch_size=512):
    backbone.eval()
    if X.ndim == 4: X = X.squeeze(-1)
    X = X.astype(np.float32)
    if X.max() > 1.0: X /= 255.0

    normalize = T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    feats = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            batch = X[i:i+batch_size]
            t = torch.from_numpy(batch).unsqueeze(1)
            t = t.repeat(1, 3, 1, 1)
            t = normalize(t)
            t = t.to(device)
            feats.append(backbone(t).cpu().numpy())
    return np.concatenate(feats, axis=0)

print('Extracting features from teacher backbone ...')
features = extract_features(teacher_backbone, X_all)
print(f'features shape : {features.shape}')

feat_path = f'{RESULT_DIR}/{model_name}_dino_features.npy'
np.save(feat_path, features)
print(f'Saved → {feat_path}')


Extracting features from teacher backbone ...
features shape : (3560, 128)
Saved → ./result_dir/261106_2034_dino/culture-noAug_dino_features.npy


In [13]:
# ─── Phase 4: Save all artifacts ──────────────────────────────────────────────
import joblib
from sklearn.preprocessing import StandardScaler

scaler_sk   = StandardScaler()
feat_scaled = scaler_sk.fit_transform(features)
joblib.dump(scaler_sk, f'{RESULT_DIR}/{model_name}_scaler.pkl')

np.save(f'{RESULT_DIR}/{model_name}_feat_scaled.npy',  feat_scaled)
np.save(f'{RESULT_DIR}/{model_name}_y_all.npy',        y_all)
np.save(f'{RESULT_DIR}/{model_name}_images_all.npy',   X_all)

torch.save({
    'teacher_backbone_state': teacher_backbone.state_dict(),
    'student_backbone_state': student_backbone.state_dict(),
    'student_head_state'    : student_head.state_dict(),
    'teacher_head_state'    : teacher_head.state_dict(),
    'optimizer_state'       : optimizer.state_dict(),
    'loss_history'          : loss_history,
    'config': {
        'IMAGE_SIZE': IMAGE_SIZE, 'BATCH_SIZE': BATCH_SIZE,
        'EPOCHS': EPOCHS, 'LR': LR, 'FEAT_DIM': FEAT_DIM,
        'OUT_DIM': OUT_DIM, 'HEAD_BOTTLE': HEAD_BOTTLE,
        'N_GLOBAL': N_GLOBAL, 'N_LOCAL': N_LOCAL,
        'STUDENT_TEMP': STUDENT_TEMP, 'TEACHER_TEMP': TEACHER_TEMP,
        'MOMENTUM_START': MOMENTUM_START, 'MOMENTUM_END': MOMENTUM_END,
    }
}, f'{RESULT_DIR}/{model_name}_dino_checkpoint.pt')

meta = {
    'result_dir'    : RESULT_DIR,
    'model_name'    : model_name,
    'dt_str'        : dt_str,
    'features_shape': list(features.shape),
    'n_classes'     : int(len(np.unique(y_all))),
    'OUT_DIM'       : OUT_DIM,
    'HEAD_BOTTLE'   : HEAD_BOTTLE,
}
with open(f'{RESULT_DIR}/meta.json', 'w') as f:
    _json.dump(meta, f, indent=2)

print('All artifacts saved.')
print(f'  RESULT_DIR: {RESULT_DIR}')
for p in Path(RESULT_DIR).iterdir():
    print(f'    {p.name}  ({p.stat().st_size:,} bytes)')


All artifacts saved.
  RESULT_DIR: ./result_dir/261106_2034_dino
    checkpoint_best.pth  (160,183,092 bytes)
    checkpoint_latest.pth  (160,183,730 bytes)
    culture-noAug_dino_checkpoint.pt  (160,187,494 bytes)
    culture-noAug_dino_features.npy  (1,822,848 bytes)
    culture-noAug_feat_scaled.npy  (1,822,848 bytes)
    culture-noAug_images_all.npy  (320,400,128 bytes)
    culture-noAug_scaler.pkl  (3,687 bytes)
    culture-noAug_y_all.npy  (14,368 bytes)
    dino_loss_curve.tif  (2,742,446 bytes)
    meta.json  (225 bytes)
